# CT-FM/CT-FM-SNP reproducible workflow

This notebook provides the codes for CT-FM/CT-FM-SNP analyses done in our preprint (https://doi.org/10.1101/2024.05.17.24307556)

1. Installation
2. Reference downloads
3. GWAS summary statistic download and organization
4. S-LDSC runs
5. CT-FM runs
6. CT-FM-SNP runs

Make sure to have anaconda/conda and mamba installed in your environment 
https://www.anaconda.com/download 
https://mamba.readthedocs.io/en/latest/installation/mamba-installation.html

## 1. Installation

Install CT-FM conda environment and LDSC

In [ ]:
git clone https://github.com/ArtemKimUSC/CTFM/
cd install 
conda env create --file ctfm3.yml
conda activate ctfm3
conda install bioconda::bedtools
git clone https://github.com/svdorn/ldsc-2.0.1.git
cd ldsc-2.0.1
pip install .

## 2. Test the conda environment

These checks verify that LDSC and SuSiE are available.

In [ ]:

conda activate ctfm3

python ldsc.py -h
R
library(susieR)
q()
conda deactivate

## 3. Download reference files

Downloading 1000G reference files and S-LDSC annotations.

In [ ]:
cd CTFM
bash install/download_reference.sh
bash install/download_ctfmsnp_reference.sh

## 4. Download GWAS summary statistics and organize the 63 traits

In [ ]:
cd CTFM
mkdir -p sumstats_ready
cd sumstats_ready

wget -O sumstats_indep107.tgz "https://zenodo.org/records/10515792/files/sumstats_indep107.tgz?download=1"
tar -xzf sumstats_indep107.tgz

while read line; do
  mv "sumstats_107/${line}.sumstats.gz" .
done < traits_63.txt

## 5. Run S-LDSC
Obtain S-LDSC heritbaility estimates for 924 annotations for each of the 63 traits.

In [ ]:
cd CTFM
conda activate ctfm3

while read -r line; do
  bash scripts/2_launch_SLDSC_default.sh "$line"
done < sumstats_ready/traits_63.txt

conda deactivate

## 6. Run CT-FM

In [ ]:
cd CTFM
dir="$(pwd)"
conda activate ctfm3

while read -r line; do
  Rscript scripts/3_launch_susie_default.R "$dir" "$line"
done < sumstats_ready/traits_63.txt

conda deactivate

## 7. Run CT-FM-SNP

### 7A. Get overlapping annotations for SNPs

In [ ]:
cd CTFM
conda activate ctfm3

while read -r line; do
  mkdir -p out/ctfmsnp/"$line"
  bash scripts/4_CTFMSNP_default_annots.sh     "data/UKBB/${line}_finemapped_snps_out.bed"     "out/ctfmsnp/${line}"
done < data/UKBB/list_UKBB_rs.txt

conda deactivate

### 7B. Run CT-FM-SNP

In [ ]:
cd CTFM
dir="$(pwd)"
conda activate ctfm3

while read -r line; do
  cd "out/ctfmsnp/${line}"
  for rs in *out; do
    Rscript "${dir}/scripts/3_launch_susie_default.R" "$dir" "$line"
  done
  cd "$dir"
done < data/UKBB/list_UKBB_rs.txt

conda deactivate

## Notes

- This notebook preserves the workflow used in our CT-FM paper with few minor changes (i.e., codes related to the SLURM environment were removed from the Notebook).
- If you plan to run it in Google Colab, you may need to adapt the conda installation step, since Colab does not provide `mamba`/`conda` by default.
- The CT-FM-SNP loop above mirrors your original command structure; if the downstream script expects a specific SNP identifier or output filename, you may have to parameterize that the the code.